# 04 — Pré-processamento (por fold, sem vazamento)
Passos 4–8 do roteiro. Tudo que é **aprendido** dos dados é ajustado **só no treino** do fold e aplicado, sem recalcular, em validação e teste. Código em `src/preprocess.py`.

| Etapa | Regra |
|---|---|
| `TotalCharges` ausente | `tenure == 0` → `0` (todos os ausentes são clientes ainda não cobrados); mediana do treino só como contingência |
| Duplicadas / outliers | **nada removido** (sem ID de cliente, e a EDA não achou outliers). Validação e teste nunca perdem linhas |
| Faixas (só `neural_faixas`) | `tenure`: 0-6, 7-12, 13-24, 25-48, 49+ · `MonthlyCharges`: quartis do treino |
| Categóricas (neurais) | one-hot ajustado no treino (`drop="if_binary"`, categoria nova → tudo 0) |
| Numéricas (neurais) | Min-Max ajustado no treino (sem recorte em val/teste) |
| Árvores | só imputação; categóricas ficam como texto |

**Variantes** (os slides pedem registrar o desempenho de forma evolutiva): `neural` → `neural_faixas` → `arvore`.
Quando o treino tem linhas repetidas pelo oversampling, os parâmetros são ajustados nas linhas **originais** (`repetido == False`).

In [ ]:
# Localiza a raiz do repositório (funciona no VS Code e no Colab) e a coloca no sys.path
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Raiz do projeto:", ROOT)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src import config as cfg
from src.preprocess import run_all, verify_preprocessing, load_preprocessed, fit_fold, variant_dir

cfg.ensure_dirs()
print("Variantes:", {k: v for k, v in cfg.PREPROC_VARIANTS.items()})

## 1. Gerar todas as variantes e folds
Salva em `data/processed/<variante>/` (`X_*` e `params_fold*.json`). Os `y_*` continuam os do particionamento.

In [ ]:
resumo = run_all()
resumo.to_csv(cfg.REPORTS_TABLES / "preprocessamento_resumo.csv", index=False)
resumo.pivot_table(index=["variante", "fold"], columns="conjunto", values="linhas").join(
    resumo.groupby(["variante", "fold"])["colunas"].first())

## 2. Verificações (levantam erro se falharem)
- nº de linhas igual ao do particionamento (nada removido) e colunas idênticas em treino/val/teste;
- nenhum valor ausente;
- min/max do Min-Max **recalculados de forma independente** a partir só das linhas originais do treino (prova que val/teste não influenciaram);
- no treino original os numéricos ficam exatamente em [0, 1];
- one-hot binário e nenhuma categoria nova em val/teste.

In [ ]:
fora = verify_preprocessing()
print("Todas as verificações passaram.")
print("\n% de linhas com algum numérico fora de [0,1] (esperado: pequeno; nada é recortado):")
fora.pivot_table(index=["variante", "fold"], columns="conjunto", values="pct_fora_0_1")

## 3. Parâmetros aprendidos no treino do Fold 1
Auditáveis em `data/processed/<variante>/params_fold1.json`.

In [ ]:
import json
p = json.loads((variant_dir("neural_faixas") / "params_fold1.json").read_text(encoding="utf-8"))
print("Min-Max:", p["minmax"])
print("Quartis de MonthlyCharges (faixas):", p["MonthlyCharges_quartis"])
print("Regra TotalCharges:", p["regra_TotalCharges"], "| mediana de contingência:", p["TotalCharges_mediana_contingencia"])
print("Colunas após one-hot (neural_faixas):", pd.read_csv(variant_dir("neural_faixas") / "X_treino_fold1.csv").shape[1])

## 4. Antes × depois (Fold 1, treino)

In [ ]:
from src.split_data import load_partition
raw, _ = load_partition(1, "treino")
X, y = load_preprocessed(1, "treino", "neural")
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for j, c in enumerate(cfg.NUMERIC_COLS):
    axes[0, j].hist(raw[c].dropna(), bins=30, color="tab:gray"); axes[0, j].set_title(f"{c} (bruto)")
    axes[1, j].hist(X[c], bins=30, color="tab:blue");          axes[1, j].set_title(f"{c} (Min-Max)")
fig.tight_layout(); fig.savefig(cfg.REPORTS_FIGURES / "04_antes_depois_minmax.png", dpi=130); plt.show()
print("X:", X.shape, "| y:", y.shape, "| % churn no treino (balanceado):", round(100 * y.mean(), 2))
X.head()

## 5. Como usar nos experimentos
```python
from src.preprocess import load_preprocessed
X_tr, y_tr = load_preprocessed(1, "treino",    "neural")        # MLP/KAN/Transformers: one-hot + Min-Max
X_va, y_va = load_preprocessed(1, "validacao", "neural")
X_te, y_te = load_preprocessed(1, "teste",     "neural")
# "neural_faixas": mesmas colunas + faixas | "arvore": Random Forest/XGBoost (categóricas como texto)
```
**Árvores:** a variante `arvore` mantém as categóricas como texto; no modelo, converter com `OrdinalEncoder` dentro do pipeline ou `dtype="category"` (XGBoost). Os ajustes devem usar só o treino.